# Lab 3 — Teaching a small model a new job: fine-tuning with LoRA

**Artificial Intelligence · 4th year · UIR · Pr. Hakim Hafidi**

| | |
|---|---|
| **Duration** | 2 hours, individual work |
| **Runtime** | **GPU required** — Runtime → Change runtime type → T4 GPU |
| **You will learn** | how supervised learning works on a language model · how to measure the gain of fine-tuning honestly · when a better prompt is enough and when it is not |
| **Deliverable** | this notebook, executed, + the `lab03_<your id>.json` file it generates, both submitted on connect.uir.ac.ma |
| **Grading** | 4 points: it runs (1) · measurements are correct (1) · interpretation (1) · critical analysis (1) |

> Labs 1 and 2 used a model **as it is**: you turned its knobs from the outside.
> Today you change the model itself. You will give it 800 labelled examples of one task
> and measure exactly what that buys you — against a well-written prompt, not against nothing.

**The rule of this lab:** every number is measured on the **same test set**, which the model
never trains on. That is the whole idea of supervised learning, and it is what makes a
comparison honest.

---
## Step 0 — Check your runtime (2 minutes)

This lab trains a real model. Without a GPU it will take hours instead of minutes.

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. Run the two cells below. The first installs the libraries (~2 minutes), the second
   refuses to continue if no GPU is active.

*No API key is needed today. Everything runs inside this Colab machine.*

In [ ]:
# Minimum versions, not exact ones: Colab already ships a recent transformers stack, and
# forcing older versions would downgrade huggingface-hub and break other Colab packages.
%pip install -q -U "transformers>=5.3" "peft>=0.19" "trl>=1.0" "datasets>=4.7" "accelerate>=1.4"

# Colab ships torchao 0.10, which is older than the minimum PEFT accepts — and PEFT raises an
# error instead of ignoring it. This lab does not use torchao, so we remove it.
%pip uninstall -q -y torchao
print("done — if Colab asks you to restart the session, do it, then re-run this cell.")

In [ ]:
import torch, time
assert torch.cuda.is_available(), (
    "No GPU. Runtime → Change runtime type → T4 GPU → Save, then re-run this cell.")
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"   memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
LAB_START = time.time()

In [ ]:
# What you actually got. Write these down: if the lab misbehaves later in the semester,
# this is the first thing to compare.
import importlib.metadata as _md, subprocess, sys
import transformers, peft, trl, datasets, huggingface_hub as hub

# Safety net: if torchao is still here and too old, PEFT will crash when building the adapter.
try:
    _v = _md.version("torchao")
    if tuple(int(x) for x in _v.split(".")[:2]) < (0, 16):
        print(f"removing torchao {_v} (older than PEFT requires, unused here)…")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
except _md.PackageNotFoundError:
    pass

VERSIONS = {"transformers": transformers.__version__, "peft": peft.__version__,
            "trl": trl.__version__, "datasets": datasets.__version__, "huggingface_hub": hub.__version__}
print(VERSIONS)

In [ ]:
#@title ⚙️ Lab 3 toolbox — run this cell once, then fold it { display-mode: "form" }
import json, time, pathlib, re, warnings
import numpy as np, pandas as pd, torch
warnings.filterwarnings("ignore")
CALL_LOG = []          # no external API in this lab
GPU_SECONDS = {}       # filled in when you train

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

# The ten intents you will work with. They are deliberately close to each other:
# telling them apart is the actual difficulty of the task.
LABELS = ["card_arrival", "card_delivery_estimate", "card_not_working", "card_swallowed",
          "declined_card_payment", "lost_or_stolen_card", "pin_blocked",
          "request_refund", "cancel_transfer", "balance_not_updated"]

def load_model():
    """Load the base model and its tokenizer once, in half precision."""
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    try:                      # transformers >= 5
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="cuda")
    except TypeError:         # transformers 4.x used the old name
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="cuda")
    model.config.pad_token_id = tok.pad_token_id or tok.eos_token_id
    return model, tok

@torch.no_grad()
def classify(model, tok, texts, system, batch_size=32, max_new_tokens=12):
    """Ask the model to label each text. Returns the raw strings it produced.

    The same function is used for every condition of this lab — base model, better prompt,
    fine-tuned model — so that the comparison is fair."""
    tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    out, t0 = [], time.time()
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i + batch_size]
        prompts = [tok.apply_chat_template(
            [{"role": "system", "content": system}, {"role": "user", "content": t}],
            tokenize=False, add_generation_prompt=True) for t in chunk]
        enc = tok(prompts, return_tensors="pt", padding=True, truncation=True, max_length=256).to("cuda")
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.pad_token_id)
        for j in range(len(chunk)):
            out.append(tok.decode(gen[j][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip())
    return out, round(time.time() - t0, 1)

def score(raw_outputs, gold, latency=None, name=""):
    """Three numbers per condition: accuracy, validity, and answer length.

    valid_rate = share of answers that are EXACTLY one of the ten labels. A model can be
    helpful and still unusable by a program if it answers in sentences."""
    clean = [o.strip().strip('."\' ').split("\n")[0].strip() for o in raw_outputs]
    valid = [c in LABELS for c in clean]
    correct = [c == g for c, g in zip(clean, gold)]
    r = {"condition": name,
         "accuracy": round(float(np.mean(correct)), 3),
         "valid_rate": round(float(np.mean(valid)), 3),
         "mean_chars": round(float(np.mean([len(o) for o in raw_outputs])), 1)}
    if latency is not None:
        r["seconds_for_test_set"] = latency
    return r

def show_examples(texts, gold, raw, n=5):
    """Look at what the model actually wrote — always do this before trusting a score."""
    for i in range(min(n, len(texts))):
        mark = "✅" if raw[i].strip() == gold[i] else "❌"
        print(f"{mark} {texts[i][:62]:<64} gold={gold[i]:<24} model={raw[i][:40]!r}")

def confusion(raw, gold, labels=None):
    """Which intents get mixed up with which."""
    from sklearn.metrics import confusion_matrix
    labels = labels or LABELS
    clean = [o.strip().split("\n")[0].strip() for o in raw]
    clean = [c if c in labels else "OTHER" for c in clean]
    cols = labels + ["OTHER"]
    cm = confusion_matrix(gold, clean, labels=cols)
    return pd.DataFrame(cm, index=[f"true:{l}" for l in cols], columns=[f"pred:{l}" for l in cols])

def folder_size_mb(path):
    return round(sum(f.stat().st_size for f in pathlib.Path(path).rglob("*") if f.is_file()) / 1e6, 2)

def export_lab(lab_id, student_id, answers: dict, tables: dict = None, extra: dict = None):
    """Build the file you submit on connect.uir.ac.ma."""
    student_id = str(student_id).strip()
    assert student_id and student_id.lower() not in ("", "xxxxx", "your_id"), \
        "Set STUDENT_ID to your real apogee number."
    payload = {"lab": lab_id, "student_id": student_id,
               "answers": {k: str(v).strip() for k, v in answers.items()},
               "tables": {k: (v.to_dict("records") if hasattr(v, "to_dict") else v)
                          for k, v in (tables or {}).items()},
               "extra": extra or {},
               "call_log": CALL_LOG,
               "created": time.strftime("%Y-%m-%d %H:%M")}
    name = f"{lab_id}_{student_id}.json"
    pathlib.Path(name).write_text(json.dumps(payload, indent=2, ensure_ascii=False))
    empty = [k for k, v in payload["answers"].items() if len(v) < 40]
    print(f"📦 {name} written ({pathlib.Path(name).stat().st_size} bytes)")
    if empty: print(f"⚠️  These answers look too short: {empty}")
    else:     print("✅ All answers filled.")
    print("\nSubmit BOTH files on connect.uir.ac.ma:")
    print(f"   1. this notebook (File → Download → .ipynb)\n   2. {name}")
    return payload

print("✅ Lab 3 toolbox loaded: load_model, classify, score, show_examples, confusion")
print(f"   Model: {MODEL_ID}   |   {len(LABELS)} intents")

---
## Part A — The task and the split (15 minutes)

The task: a customer writes one sentence to their bank, and the system must output **one of ten
intents** — exactly, as a machine-readable label.

```
"My card still hasn't arrived after two weeks"   →   card_arrival
"The ATM kept my card"                            →   card_swallowed
```

These ten intents are close on purpose. `card_arrival` and `card_delivery_estimate` are both
about a card in the post; `card_not_working` and `declined_card_payment` both describe a
payment that failed. A model has to be precise, not just plausible.

Three pieces of data, and they never mix:

| | Size | What it is for |
|---|---|---|
| **train** | up to 800 | the examples the model learns from |
| **validation** | 200 | to check training is going well, while it runs |
| **test** | 300 | measured **once per condition**, never trained on |

Your split is seeded with your apogee number, so your numbers are yours alone.

In [ ]:
STUDENT_ID = "XXXXX"   # TODO: your apogee number — it also seeds YOUR split

SEED = int("".join(c for c in STUDENT_ID if c.isdigit())[-4:] or "0")
assert SEED > 0, "Set STUDENT_ID to your apogee number first."
print(f"Your personal seed: {SEED}")

In [ ]:
from datasets import load_dataset, concatenate_datasets

# mteb/banking77 is the parquet mirror of the original dataset. The original PolyAI/banking77
# ships a loading script, which `datasets` 4.x no longer runs.
# (The "unauthenticated requests to the HF Hub" warning is harmless — the data is public.)
parts = load_dataset("mteb/banking77")
raw = concatenate_datasets([parts[k] for k in parts])   # we make our own split below
print(f"{len(raw)} messages over {len(set(raw['label_text']))} intents in the source")

keep = raw.filter(lambda ex: ex["label_text"] in LABELS)
keep = keep.rename_column("label_text", "intent").shuffle(seed=SEED)

n_test, n_val = 300, 200
n_train = min(800, len(keep) - n_test - n_val)
assert n_train >= 400, f"Only {len(keep)} examples for these ten intents — too few."
test  = keep.select(range(n_test))
val   = keep.select(range(n_test, n_test + n_val))
train = keep.select(range(n_test + n_val, n_test + n_val + n_train))

TEST_TEXTS = test["text"]
TEST_GOLD  = test["intent"]

print(f"train {len(train)} · val {len(val)} · test {len(test)}")
print("\nThree examples from YOUR training set:")
for i in range(3):
    print(f"  {train[i]['text'][:70]!r:<74} → {train[i]['intent']}")
print("\nIntent counts in your test set:")
print(pd.Series(TEST_GOLD).value_counts().to_string())

---
## Part B — Two prompts, before any training (20 minutes)

Before training anything, find out how far a **prompt** gets you. This is the comparison that
makes the rest of the lab honest: if a better prompt already solves the task, fine-tuning is a
waste of GPU time.

**Condition A** — a naive instruction, the kind everyone writes first.
**Condition B** — the same model, with a prompt that names the ten labels, forbids explanations,
and shows two worked examples (few-shot).

In [ ]:
model, tok = load_model()
print(f"loaded {MODEL_ID}")
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f} million")

In [ ]:
SYSTEM_A = "Classify the customer message into an intent."

raw_A, sec_A = classify(model, tok, TEST_TEXTS, SYSTEM_A)
res_A = score(raw_A, TEST_GOLD, sec_A, "A · base model, naive prompt")
print(res_A)
print("\nWhat it actually answered:")
show_examples(TEST_TEXTS, TEST_GOLD, raw_A)

In [ ]:
# TODO (5–10 lines): write a better prompt for the SAME model.
#   It must: (1) list the ten allowed labels — the variable LABELS holds them,
#            (2) forbid any explanation or punctuation,
#            (3) show two worked examples (few-shot), taken from the TRAINING set only.
#   Everything you learned in Lab 2 about `system` instructions applies here.

SYSTEM_B = f"""
...
"""

assert len(SYSTEM_B) > 200, "Condition B should be a real prompt, not one line."
assert all(l in SYSTEM_B for l in LABELS), "Your prompt must list the ten labels."

raw_B, sec_B = classify(model, tok, TEST_TEXTS, SYSTEM_B)
res_B = score(raw_B, TEST_GOLD, sec_B, "B · base model, engineered prompt")
print(res_B)
print("\nWhat it actually answered:")
show_examples(TEST_TEXTS, TEST_GOLD, raw_B)

In [ ]:
print(pd.DataFrame([res_A, res_B]).to_string(index=False))
assert res_B["valid_rate"] >= res_A["valid_rate"], \
    "Your engineered prompt should at least improve the share of valid labels — try again."
print("\n✅ Part B done. Note both numbers: they are what fine-tuning must beat.")

---
## Part C — Fine-tuning with LoRA (30 minutes)

Now supervised learning, on a language model. You show it 800 pairs (message → label) and it
adjusts its parameters to reduce its error — the same loop as any other model in this course.

**LoRA** (Low-Rank Adaptation) changes one thing: instead of updating all 500 million
parameters, it freezes them and trains a small set of extra weights alongside. You will see at
the end how small. That is why this fits on a free Colab GPU.

Each training example is written as a conversation whose answer is the label and nothing else.
The model is trained to produce exactly that.

In [ ]:
SYSTEM_FT = "Classify the customer message into one intent label."

def to_chat(ex):
    return {"messages": [{"role": "system",  "content": SYSTEM_FT},
                         {"role": "user",    "content": ex["text"]},
                         {"role": "assistant","content": ex["intent"]}]}

train_chat = train.map(to_chat, remove_columns=train.column_names)
val_chat   = val.map(to_chat,   remove_columns=val.column_names)
print(json.dumps(train_chat[0]["messages"], indent=2)[:400])

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# TODO (2 lines): set the two training choices below.
#   EPOCHS      : how many passes over the 800 examples. Start with 3.
#   LEARN_RATE  : how big a step at each update. 2e-4 is the usual value for LoRA.
EPOCHS = ...
LEARN_RATE = ...

assert isinstance(EPOCHS, int) and 1 <= EPOCHS <= 5, "Pick an integer number of epochs (3 is a good start)."
assert isinstance(LEARN_RATE, float), "LEARN_RATE should be a float, e.g. 2e-4."

peft_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])

cfg = SFTConfig(output_dir="/content/lora_800", num_train_epochs=EPOCHS,
                per_device_train_batch_size=8, gradient_accumulation_steps=2,
                learning_rate=LEARN_RATE, logging_steps=10,
                eval_strategy="epoch", save_strategy="no", report_to="none",
                max_length=256, fp16=True, seed=SEED)

t0 = time.time()
trainer = SFTTrainer(model=MODEL_ID, args=cfg, train_dataset=train_chat,
                     eval_dataset=val_chat, peft_config=peft_cfg)
trainer.train()
GPU_SECONDS["train_800"] = round(time.time() - t0, 1)
trainer.save_model("/content/lora_800")
print(f"\n⏱️  trained in {GPU_SECONDS['train_800']} s on {len(train)} examples")

In [ ]:
# Two losses, epoch by epoch:
#   train_loss — the error on the 800 examples the model is learning from
#   eval_loss  — the error on the 200 validation examples it never trains on
# (Training loss is logged every 10 steps, validation only at the end of each epoch;
#  the cell below lines them up so you can compare them.)
import matplotlib.pyplot as plt

hist = pd.DataFrame(trainer.state.log_history)
tr = hist.dropna(subset=["loss"]).copy()
ev = hist.dropna(subset=["eval_loss"]).copy()
tr["ep"] = np.ceil(tr["epoch"]).astype(int)
ev["ep"] = ev["epoch"].round().astype(int)
curve = pd.concat([tr.groupby("ep")["loss"].mean().rename("train_loss"),
                   ev.groupby("ep")["eval_loss"].mean().rename("eval_loss")], axis=1).round(4)
curve.index.name = "epoch"
print(curve.reset_index().to_string(index=False))

plt.figure(figsize=(6, 3.2))
plt.plot(tr["epoch"], tr["loss"], label="training loss", alpha=0.6)
plt.plot(ev["epoch"], ev["eval_loss"], "o-", label="validation loss")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("Is it still learning?")
plt.tight_layout(); plt.show()

if curve["eval_loss"].iloc[-1] <= curve["eval_loss"].min() + 1e-6:
    print("Validation loss is still going down — the model has not started overfitting yet.")
else:
    print("Validation loss has started rising — from here on, the model is memorising.")

adapter_mb = folder_size_mb("/content/lora_800")
full_mb = round(sum(p.numel() * 2 for p in model.parameters()) / 1e6, 1)
print(f"\nLoRA adapter on disk : {adapter_mb} MB")
print(f"Full model in memory : {full_mb} MB")
print(f"Ratio                : {full_mb / adapter_mb:.0f}×")

---
## Part D — Condition C, and how much data you really need (20 minutes)

Measure the fine-tuned model on the **same 300 test messages**, with the same `classify()`
function. Nothing else changes, so the comparison holds.

In [ ]:
from peft import PeftModel

# One base model, several adapters: load the first one under a name, so that we can add the
# second one later and switch between them. This is what makes LoRA cheap to serve.
ft_model = PeftModel.from_pretrained(model, "/content/lora_800", adapter_name="ft800").eval()
ft_model.set_adapter("ft800")

# TODO (2 lines): run classify() on the fine-tuned model, then score it.
#   Use SYSTEM_FT — the same system string the model was trained with — and name it
#   "C · fine-tuned, 800 examples".
raw_C, sec_C = ...
res_C = ...

print(res_C)
print("\nWhat it actually answered:")
show_examples(TEST_TEXTS, TEST_GOLD, raw_C)

In [ ]:
comparison = pd.DataFrame([res_A, res_B, res_C])
print(comparison.to_string(index=False))
print()
print("gain of fine-tuning over the engineered prompt:")
print(f"  accuracy   {res_B['accuracy']:.3f} → {res_C['accuracy']:.3f}   "
      f"({(res_C['accuracy'] - res_B['accuracy']) * 100:+.1f} points)")
print(f"  valid_rate {res_B['valid_rate']:.3f} → {res_C['valid_rate']:.3f}   "
      f"({(res_C['valid_rate'] - res_B['valid_rate']) * 100:+.1f} points)")

In [ ]:
# Where does the fine-tuned model still fail? Look at the pairs it confuses.
cm = confusion(raw_C, TEST_GOLD)
errors = []
for t in cm.index:
    for p in cm.columns:
        if t.replace("true:", "") != p.replace("pred:", "") and cm.loc[t, p] > 0:
            errors.append({"true": t.replace("true:", ""), "predicted": p.replace("pred:", ""),
                           "count": int(cm.loc[t, p])})
top_confusions = pd.DataFrame(errors).sort_values("count", ascending=False).head(5)
print("The five most frequent confusions:")
print(top_confusions.to_string(index=False))

### D bis — The same thing with 50 examples

How much of that gain came from the **size** of the training set? Retrain on 50 examples only —
about two minutes — and measure again.

In [ ]:
small_train = train.select(range(50)).map(to_chat, remove_columns=train.column_names)

cfg_small = SFTConfig(output_dir="/content/lora_50", num_train_epochs=EPOCHS,
                      per_device_train_batch_size=8, gradient_accumulation_steps=2,
                      learning_rate=LEARN_RATE, logging_steps=5, save_strategy="no",
                      report_to="none", max_length=256, fp16=True, seed=SEED)

t0 = time.time()
trainer50 = SFTTrainer(model=MODEL_ID, args=cfg_small, train_dataset=small_train, peft_config=peft_cfg)
trainer50.train()
GPU_SECONDS["train_50"] = round(time.time() - t0, 1)
trainer50.save_model("/content/lora_50")

# Add the second adapter to the SAME base model and switch to it — no second model in memory.
ft_model.load_adapter("/content/lora_50", adapter_name="ft50")
ft_model.set_adapter("ft50")
raw_D, sec_D = classify(ft_model, tok, TEST_TEXTS, SYSTEM_FT)
ft_model.set_adapter("ft800")   # switch back, so the notebook ends in a known state
res_D = score(raw_D, TEST_GOLD, sec_D, "D · fine-tuned, 50 examples")

comparison = pd.DataFrame([res_A, res_B, res_C, res_D])
print(comparison.to_string(index=False))
print(f"\nGPU time: {GPU_SECONDS}")

In [ ]:
assert len(comparison) == 4, "You need all four conditions in the table."
assert res_C["valid_rate"] >= 0.9, \
    "A fine-tuned model should almost always output a valid label — check SYSTEM_FT is the one used."
print("✅ Part D done. Look hard at the D row before answering the questions.")

---
## Part E — Analysis (20 minutes)

Answer in English, in your own words, **with your own numbers**. Two or three sentences each.
This is where most of the grade is.

In [ ]:
# Q1 — Compare conditions A and B. How much of the problem did a better PROMPT solve, on
#      accuracy and on valid_rate? Quote both numbers.
Q1 = """
...
"""

# Q2 — Now compare B and C. What exactly did fine-tuning add that the prompt could not?
#      Look at your mean_chars column too, and say what it tells you.
Q2 = """
...
"""

# Q3 — Compare C (800 examples) and D (50 examples). Which of the two metrics recovered with
#      only 50 examples, and which one needed the full 800? What does that say about the
#      difference between learning a FORMAT and learning a SKILL?
Q3 = """
...
"""

# Q4 — Look at your five most frequent confusions. Pick one pair and explain, by reading the
#      messages, why those two intents are hard to tell apart. Would more data fix it?
Q4 = """
...
"""

# Q5 — Critical analysis. You now have three ways to get this job done: a prompt, a fine-tuned
#      model, or a large commercial model called through an API (Labs 1–2). For a Moroccan bank
#      running this on 10,000 messages a day, argue for ONE of them, using your measured
#      numbers (accuracy, latency, adapter size, GPU time) and naming at least one criterion
#      your measurements do NOT cover.
Q5 = """
...
"""

In [ ]:
answers = {"Q1": Q1, "Q2": Q2, "Q3": Q3, "Q4": Q4, "Q5": Q5}
problems = []
if STUDENT_ID.strip().upper() in ("XXXXX", ""): problems.append("STUDENT_ID is not filled in")
for k, v in answers.items():
    if len(v.strip()) < 120: problems.append(f"{k} is too short (write 2–3 sentences)")
    if "..." in v:           problems.append(f"{k} still contains the placeholder")
if problems:
    print("⚠️  Not ready to submit:"); [print("   -", p) for p in problems]
else:
    print("✅ Everything looks complete. Run the export cell below.")

---
## Submit

Run the cell below, then upload **both** files on connect.uir.ac.ma.

In [ ]:
print(f"total lab time: {(time.time() - LAB_START) / 60:.0f} minutes")
print(f"GPU training time: {sum(GPU_SECONDS.values()):.0f} s")
print()
export_lab("lab03", STUDENT_ID,
           answers={"Q1": Q1, "Q2": Q2, "Q3": Q3, "Q4": Q4, "Q5": Q5},
           tables={"comparison": comparison,
                   "loss_curve": curve.reset_index(),
                   "top_confusions": top_confusions},
           extra={"seed": SEED, "epochs": EPOCHS, "learning_rate": LEARN_RATE,
                  "system_B": SYSTEM_B, "gpu_seconds": GPU_SECONDS,
                  "adapter_mb": adapter_mb, "full_model_mb": full_mb,
                  "versions": VERSIONS})

---
## Going further (optional, not graded)

- Train for 10 epochs instead of 3 and watch `eval_loss` in the curve. At which epoch does it
  stop improving, and what is `loss` doing meanwhile? You have just seen overfitting on an LLM.
- Set `r=4` instead of `r=16` in the LoRA config. How much smaller is the adapter, and how much
  accuracy do you lose?
- Take your ten worst test messages and pass them to the **base** model with condition B's
  prompt. Does it get any of them right? What does that tell you about where the 800 examples
  actually helped?
- Merge the adapter into the base model (`ft_model.merge_and_unload()`) and time a single
  prediction before and after. Why is a merged model faster to serve?